# Attention mecanism and Transformers

This notebook implements the foundational architecture of a Transformer, starting with the attention mechanism. The model relies entirely on dense and attention layers, doing away with recurrent and convolutional networks.

## 1. Imports

In [ ]:
import torch
import math

## 2. Data

I define dummy variables here that will be used across all cells to test my functions

In [ ]:
batch_size = 2
seq_length = 5
d_model = 64 # Embedding dimension

print(f"Test parameters initialized: batch_size={batch_size}, seq_length={seq_length}, d_model={d_model}")

Test parameters initialized: batch_size=2, seq_length=5, d_model=64


## 3.Scaled Dot-Product Attention

Attention is computed using Query vectors (to match others), Key vectors (to be matched), and Value vectors (the information to extract).
The mathematical formulation is:

$Attention(x) = softmax(\frac{QK^T}{\sqrt{d_k}})V$.

The dot product of Q and K is scaled by dividing by $\sqrt{d_k}$ (the dimension of the queries/keys) before applying the softmax function to obtain the attention scores.

In [ ]:
# === Function ===
def scaled_dot_product_attention(Q, K, V, mask=None):
  # d_k : dimension of Q and K
  d_k = Q.size(-1)

  # 1. Dot product Q*K^T
  scores = torch.matmul(Q,K.transpose(-2,-1))

  # 2. Scale
  scores = scores / math.sqrt(d_k)

  # Optional : mask
  if mask is not None:
    scores = scores.masked_fill(mask == 0, -1e9)

  # 3. Softmax
  attention_weights = torch.nn.functional.softmax(scores,dim=-1)

  # 4. Dot product with V
  output = torch.matmul(attention_weights, V)

  return output, attention_weights

# === Test ===
print("=== Test Scaled Dot-Product Attention ===")
d_k_test = 8
Q_test = torch.randn(batch_size, seq_length, d_k_test)
K_test = torch.randn(batch_size, seq_length, d_k_test)
V_test = torch.randn(batch_size, seq_length, d_k_test)

output_attn, weights = scaled_dot_product_attention(Q_test, K_test, V_test)
print(f"Output shape: {output_attn.shape} (Expected: torch.Size([{batch_size}, {seq_length}, {d_k_test}]))")
print(f"Attention weights shape: {weights.shape} (Expected: torch.Size([{batch_size}, {seq_length}, {seq_length}]))")


=== Test Scaled Dot-Product Attention ===
Output shape: torch.Size([2, 5, 8]) (Expected: torch.Size([2, 5, 8]))
Attention weights shape: torch.Size([2, 5, 5]) (Expected: torch.Size([2, 5, 5]))


## 4. Multi-Headed Self-Attention

To overcome the limited capacity of a single attention head, multi-headed attention allows the model to jointly attend to information from different representation subspaces. Each head operates in a reduced dimension ($d/h$). The results from all heads are concatenated and passed through a final linear projection.

In [ ]:
# === Function ===
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        # The dimension of each head is d / h
        self.d_k = d_model // num_heads

        self.W_q = torch.nn.Linear(d_model, d_model)
        self.W_k = torch.nn.Linear(d_model, d_model)
        self.W_v = torch.nn.Linear(d_model, d_model)

        self.W_o = torch.nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)

        # Linear projections and split into num_heads
        q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # Apply dot-product attention
        output, _ = scaled_dot_product_attention(q, k, v, mask)

        # Concatenate all heads
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        # Final linear projection
        return self.W_o(output)



# === TEST ===
print("\n=== Test Multi-Head Attention ===")
num_heads = 8
# Create a random input tensor (simulating the word embeddings)
x_test = torch.randn(batch_size, seq_length, d_model)

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
# Self-attention: Q, K, and V are identical
output_mha = mha(x_test, x_test, x_test)

print(f"Output shape: {output_mha.shape} (Expected: torch.Size([{batch_size}, {seq_length}, {d_model}]))")


=== Test Multi-Head Attention ===
Output shape: torch.Size([2, 5, 64]) (Expected: torch.Size([2, 5, 64]))


## 5. Positional Encoding

Because the model doesn't inherently process data sequentially, it doesn't know word positions or orderings. To fix this, vectors that uniquely encode each position are added to the input. Absolute positional embeddings are created using sine and cosine functions: $sin(\omega_k \cdot t)$ for even indices and $cos(\omega_k \cdot t)$ for odd indices, where $\omega_k = \frac{1}{10000^{2k/d}}$.  

In [ ]:
# === Function ===
class PositionalEncoding(torch.nn.Module):
  def __init__(self,d_model,max_len=5000):
    super().__init__()

    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0,max_len, dtype=torch.float).unsqueeze(1)

    # Divisor term
    div_term = torch.exp(torch.arange(0,d_model,2).float() * -(math.log(10000.0) /d_model))

    # Sine and Cosine
    pe[:,0::2] = torch.sin(position * div_term)
    pe[:,1::2] = torch.cos(position * div_term)

    pe = pe.unsqueeze(0)
    self.register_buffer('pe', pe)

  def forward(self,x):
    # Add positional encodinf to word embeddings
    x += self.pe[:,:x.size(1):]
    return x

# === TEST ===
print("\n--- Test Positional Encoding ---")
pe = PositionalEncoding(d_model=d_model)
output_pe = pe(x_test) # Using the x_test tensor from the previous cell

print(f"Output shape: {output_pe.shape} (Expected: torch.Size([{batch_size}, {seq_length}, {d_model}]))")



--- Test Positional Encoding ---
Output shape: torch.Size([2, 5, 64]) (Expected: torch.Size([2, 5, 64]))


## 6. The Self-Attention Block (Encoder)

The encoder is composed of a stack of $N$ identical blocks. Its primary role is to map an input sequence of token representations into a sequence of continuous contextualized representations.

**Multi-Head Self-Attention:** Allows the model to jointly attend to information from different representation subspaces by splitting vectors into $h$ heads:
  $$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O$$

**Feed-Forward Network (FFN):** Applies position-wise linear transformations with a non-linearity in between to add model expressivity:
  $$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

**Add & Norm (Residual Connections & Layer Normalization):** Wrapped around each sub-layer to prevent vanishing gradients in deep networks:
  $$\text{LayerNorm}(x + \text{SubLayer}(x))$$

In [ ]:
#== Function ===

class TransformerEncoderBlock(torch.nn.Module):
  def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        # Multi Head Attention
        self.self_attn = MultiHeadAttention(d_model, num_heads)

        # Feed Forward Network
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_ff),
            torch.nn.ReLU(),
            torch.nn.Linear(d_ff, d_model)
        )

        # Layer Normalization
        self.layer_norm1 = torch.nn.LayerNorm(d_model)
        self.layer_norm2 = torch.nn.LayerNorm(d_model)

        self.dropout = torch.nn.Dropout(dropout)

  def forward(self, x, mask=None):
      # Multi-Head Attention + Residual Connection + Add & Norm
      attn_output = self.self_attn(x, x, x, mask)
      x = self.layer_norm1(x + self.dropout(attn_output))

      # Feed Forward Network + Residual Connection + Add & Norm
      ffn_output = self.ffn(x)
      out = self.layer_norm2(x + self.dropout(ffn_output))

      return out


# === TEST ===
print("\n--- Test Transformer Encoder Block ---")
d_ff = 256 # Dimension of the hidden layer in the Feed-Forward Network
encoder_block = TransformerEncoderBlock(d_model=d_model, num_heads=num_heads, d_ff=d_ff)

# Pass the data with positional encoding
output_encoder = encoder_block(output_pe)

print(f"Output shape: {output_encoder.shape} (Expected: torch.Size([{batch_size}, {seq_length}, {d_model}]))")
print("The encoder block works correctly! The model is ready to stack N layers.")



--- Test Transformer Encoder Block ---
Output shape: torch.Size([2, 5, 64]) (Expected: torch.Size([2, 5, 64]))
The encoder block works correctly! The model is ready to stack N layers.


## 7. Decoder

The decoder also features a stack of $N$ identical blocks, but it is structured for autoregressive sequence generation.

* **Masked Multi-Head Self-Extension**

* **Encoder-Decoder Attention (Cross-Attention):** This sub-layer bridges the encoder and decoder. The Queries ($Q$) come from the previous decoder layer, while the Keys ($K$) and Values ($V$) come from the output of the encoder:
  $$\text{Attention}(Q_{dec}, K_{enc}, V_{enc}) = \text{softmax}\left(\frac{Q_{dec}K_{enc}^T}{\sqrt{d_k}}\right)V_{enc}$$

* **Feed-Forward & Add & Norm:** Similar to the encoder, each sub-layer is wrapped with residual connections and layer normalization:
  $$\text{LayerNorm}(x + \text{SubLayer}(x))$$

In [ ]:
# === Function ===
class TransformerDecoderBlock(torch.nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        # 1. Masked Self-Attention
        self.masked_self_attn = MultiHeadAttention(d_model, num_heads)

        # 2. Encoder-Decoder Attention
        self.encoder_decoder_attn = MultiHeadAttention(d_model, num_heads)

        # 3. Feed Forward Network
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_ff),
            torch.nn.ReLU(),
            torch.nn.Linear(d_ff, d_model)
        )

        # Layer Normalizations (Add & Norm)
        self.layer_norm1 = torch.nn.LayerNorm(d_model)
        self.layer_norm2 = torch.nn.LayerNorm(d_model)
        self.layer_norm3 = torch.nn.LayerNorm(d_model)

        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):

        # 1. Masked Self-Attention + Residual + Add & Norm
        attn_output = self.masked_self_attn(x, x, x, tgt_mask)
        x = self.layer_norm1(x + self.dropout(attn_output))

        # 2. Encoder-Decoder Attention + Residual + Add & Norm
        # Q comes from the decoder (x), K and V come from the encoder output
        cross_attn_output = self.encoder_decoder_attn(Q=x, K=encoder_output, V=encoder_output, mask=src_mask)
        x = self.layer_norm2(x + self.dropout(cross_attn_output))

        # 3. Feed Forward Network + Residual + Add & Norm
        ffn_output = self.ffn(x)
        out = self.layer_norm3(x + self.dropout(ffn_output))

        return out


# === TEST  ===
print("\n--- Test Transformer Decoder Block ---")
decoder_block = TransformerDecoderBlock(d_model=d_model, num_heads=num_heads, d_ff=d_ff)

# Simulating target sequence and encoder output
tgt_test = torch.randn(batch_size, seq_length, d_model)
encoder_output_test = torch.randn(batch_size, seq_length, d_model)

# Forward pass through the decoder block
output_decoder = decoder_block(tgt_test, encoder_output_test)

print(f"Decoder output shape: {output_decoder.shape} (Expected: torch.Size([{batch_size}, {seq_length}, {d_model}]))")
print("The decoder block works correctly with Masked Attention and Cross-Attention!")


--- Test Transformer Decoder Block ---
Decoder output shape: torch.Size([2, 5, 64]) (Expected: torch.Size([2, 5, 64]))
The decoder block works correctly with Masked Attention and Cross-Attention!


## 8. Transformer

The overall model integrates the encoder and decoder into a unified sequence-to-sequence framework:

* **Input Embeddings & Positional Encoding**

* **Workflow:** The source sequence flows through the encoder layers to generate context-aware key-value memories. The target sequence flows through the decoder, utilizing both masked self-attention and cross-attention over the encoder's representations.

* **Final Output Layer:** The continuous representations from the top of the decoder pass through a linear projection (*Linear*) followed by a *Softmax* function to yield output probabilities over the vocabulary:
  $$\text{Output Probabilities} = \text{softmax}(W_s \cdot h_t)$$

In [ ]:
# === Function ===

class Transformer(torch.nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_len=5000, dropout=0.1):
        super().__init__()

        # 1. Embeddings & Positional Encoding for Source and Target
        self.src_embedding = torch.nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = torch.nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len)

        # 2. Stack of Encoder Layers
        self.encoder_layers = torch.nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # 3. Stack of Decoder Layers
        self.decoder_layers = torch.nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # 4. Final Linear Output Projection
        self.fc_out = torch.nn.Linear(d_model, tgt_vocab_size)
        self.dropout = torch.nn.Dropout(dropout)
        self.d_model = d_model

    f_scale = math.sqrt

    def encode(self, src, src_mask=None):
        x = self.src_embedding(src) * math.sqrt(self.d_model)
        x = self.positional_encoding(x)
        x = self.dropout(x)
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        return x

    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        x = self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        x = self.positional_encoding(x)
        x = self.dropout(x)
        for layer in self.decoder_layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return x

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        encoder_output = self.encode(src, src_mask)
        decoder_output = self.decode(tgt, encoder_output, src_mask, tgt_mask)
        output = self.fc_out(decoder_output)
        return output


# === Test ===
print("\n--- Test Complete Transformer Model ---")
src_vocab_size = 1000
tgt_vocab_size = 1000
num_layers = 3

# Instantiate the complete Transformer
transformer = Transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    d_ff=d_ff
)

# Simulate source and target token batches
src_tokens = torch.randint(0, src_vocab_size, (batch_size, seq_length))
tgt_tokens = torch.randint(0, tgt_vocab_size, (batch_size, seq_length))

# Forward pass
logits = transformer(src_tokens, tgt_tokens)

print(f"Source tokens shape: {src_tokens.shape}")
print(f"Target tokens shape: {tgt_tokens.shape}")
print(f"Transformer output logits shape: {logits.shape} (Expected: torch.Size([{batch_size}, {seq_length}, {tgt_vocab_size}]))")
print("The complete Encoder-Decoder Transformer architecture is fully operational!")


--- Test Complete Transformer Model ---
Source tokens shape: torch.Size([2, 5])
Target tokens shape: torch.Size([2, 5])
Transformer output logits shape: torch.Size([2, 5, 1000]) (Expected: torch.Size([2, 5, 1000]))
The complete Encoder-Decoder Transformer architecture is fully operational!
